# T4 Exact Shared-Scale GEMM Bring-Up

Run this notebook only after `rns-pre-cuda` is green on the released package. The CPU oracle, seven-rail profiles, fixtures, and verifier are already complete. Only the two CUDA backend methods are intentionally missing.


In [ ]:
!nvidia-smi
!rm -rf /content/rns_engine
!git clone https://github.com/playfularchitect/rns_engine.git /content/rns_engine
%cd /content/rns_engine
%pip install -e .


In [ ]:
import json, platform, subprocess, time
import numpy as np
import rns_engine as rns

print('python', platform.python_version())
print('numpy', np.__version__)
print('rns_engine', rns.__version__)
report = rns.run_pre_cuda_readiness()
print(report.to_json())
report.require_ready()


In [ ]:
left = rns.SharedScaleMatrix(
    np.array([[2**42 + 17, -(2**35) + 5, 123456789],
              [-999999999, 2**40 - 3, 77]], dtype=object),
    scale=6,
)
right = rns.SharedScaleMatrix(
    np.array([[17, -(2**31) + 1],
              [2**33 + 9, 41],
              [-12345, 2**29 - 7]], dtype=object),
    scale=35,
)
config = rns.WideRNSConfig.balanced_seven_rail()
fixture = rns.build_cuda_gemm_fixture(
    't4-eight-plane-witness', left, right, config=config,
    left_plane_count=8, right_plane_count=8,
)
fixture.write_json('/content/t4_exact_gemm_fixture.json')
print('fixture written:', fixture.name)


## CUDA_BACKEND_INTEGRATION_POINT

Replace only the two method bodies below. `grouped_partials` must return `int32` coefficient matrices with shape `(left_planes + right_planes - 1, M, N)`. `weighted_rails` must return one residue matrix per requested modulus. Do not change the fixture or verifier.


In [ ]:
class T4CudaExactBackend:
    name = 't4-cuda-exact-pipeline'

    def grouped_partials(self, left_planes, right_planes):
        # TODO_CUDA_GROUPED_PARTIALS: integrate INT8 Tensor Core/cuBLASLt body here.
        raise NotImplementedError('CUDA grouped-partial kernel not integrated yet')

    def weighted_rails(self, grouped_partials, weights, moduli):
        # TODO_CUDA_WEIGHTED_RAILS: integrate GPU modular weighting/accumulation here.
        raise NotImplementedError('CUDA seven-rail kernel not integrated yet')

backend = T4CudaExactBackend()


In [ ]:
# Run only after both CUDA methods are implemented.
verification = rns.verify_backend(fixture, backend)
print(verification)
verification.require_passed()


In [ ]:
def median_seconds(callable_, repeats=21, warmups=5):
    for _ in range(warmups):
        callable_()
    samples = []
    for _ in range(repeats):
        started = time.perf_counter_ns()
        callable_()
        samples.append(time.perf_counter_ns() - started)
    return int(np.median(np.asarray(samples, dtype=np.int64))) / 1e9, samples

left_planes = np.asarray(fixture.left_planes, dtype=np.int8)
right_planes = np.asarray(fixture.right_planes, dtype=np.int8)
cpu = rns.CpuExactPipelineBackend()
cpu_seconds, cpu_samples = median_seconds(lambda: cpu.grouped_partials(left_planes, right_planes))
cuda_seconds, cuda_samples = median_seconds(lambda: backend.grouped_partials(left_planes, right_planes))
print({'cpu_seconds': cpu_seconds, 'cuda_seconds': cuda_seconds, 'speedup': cpu_seconds / cuda_seconds})
